# 02 · ¿El problema de ranking es difícil o trivial?

**Pregunta:** si el cliente diera pistas *perfectas* (sin ruido), ¿cuántas transacciones de su
historial empatan con ellas? Si casi siempre hay una sola, una regla exacta resuelve el caso y un
ranker aprendido aporta poco. Si hay muchos empates, el loop de aclaración está justificado.

Diseño:
- 20.000 transacciones objetivo del universo disputable (Purchase/Payment/Withdrawal,
  estado Approved/Pending), muestreadas de forma **estratificada proporcional por país del
  cliente × canal** (18 estratos) con un hash con semilla 42.
- Fecha de sesión = fecha de la transacción + U(0, 30 días), con semilla 42.
- Candidatas = todas las disputables del mismo cliente (cualquier estado) con fecha en
  [sesión − 60 días, sesión]. Con un retraso ≤ 30 días la objetivo siempre está en la ventana.
- "Comercio" = `merchant_name` si existe; si no, `transaction_category` (pagos); si no, el tipo
  (retiros). Es lo más fino que un cliente podría nombrar de cada transacción.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
from common import connect, SEED, DISPUTABLE_SQL, TARGET_STATUSES, sql_list, WINDOW_DAYS, MAX_SESSION_LAG_DAYS
pd.set_option("display.width", 200, "display.max_columns", 40)
con = connect()
q = lambda s: con.sql(s).df()
N = 20_000

con.execute(f"""
CREATE OR REPLACE TEMP TABLE disp AS
SELECT transaction_id, customer_id, transaction_date, amount, currency, channel, customer_country,
       transaction_type, transaction_status, coalesce(merchant_name, transaction_category, transaction_type) AS merchant_key
FROM transactions WHERE {DISPUTABLE_SQL}""")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE tgt AS
WITH elig AS (
  SELECT *, customer_country || '|' || channel AS stratum,
         row_number() OVER (PARTITION BY customer_country, channel ORDER BY hash(transaction_id, {SEED})) AS rk,
         count(*) OVER (PARTITION BY customer_country, channel) AS n_stratum
  FROM disp WHERE transaction_status IN ({sql_list(TARGET_STATUSES)}))
SELECT *, n_stratum / count(*) OVER () AS share FROM elig""")
con.execute(f"DELETE FROM tgt WHERE rk > round(share * {N})")
tgt = q("SELECT transaction_id FROM tgt ORDER BY transaction_id")
rng = np.random.default_rng(SEED)
tgt["lag_s"] = rng.uniform(0, MAX_SESSION_LAG_DAYS * 86400, len(tgt)).round()
con.register("lags", tgt)
con.execute("""CREATE OR REPLACE TEMP TABLE tgt2 AS
  SELECT t.*, t.transaction_date + to_seconds(l.lag_s::BIGINT) AS session_ts FROM tgt t JOIN lags l USING (transaction_id)""")
print(q("SELECT count(*) AS objetivos, count(DISTINCT customer_id) AS clientes, count(DISTINCT stratum) AS estratos FROM tgt2").to_string(index=False))
print(q("SELECT customer_country, channel, count(*) n FROM tgt2 GROUP BY ALL ORDER BY 1, 2").pivot(index="customer_country", columns="channel", values="n").to_string())

 objetivos  clientes  estratos
     20001     18112        18
channel            ATM   App  Branch   POS  Transfer   Web
customer_country                                          
Argentina         1194   597     119  1393        80   598
Colombia          1806   899     179  2101       121   901
México            3007  1501     299  3502       202  1502


## Candidatas por consulta y empates según la pista
Para cada objetivo se cuentan las candidatas de la ventana que cumplen cada pista perfecta
(la objetivo siempre cumple, así que "1" = candidata única = la regla resuelve sola).

In [2]:
cand = q(f"""
SELECT t.transaction_id AS tid, t.customer_country AS country, t.channel, t.merchant_key AS t_merchant, t.transaction_type AS kind,
       count(*) AS n_window,
       count(*) FILTER (WHERE c.amount = t.amount) AS amount_exact,
       count(*) FILTER (WHERE abs(c.amount - t.amount) <= 0.10 * t.amount) AS amount_10,
       count(*) FILTER (WHERE c.merchant_key = t.merchant_key) AS merchant_only,
       count(*) FILTER (WHERE abs(date_diff('second', c.transaction_date, t.transaction_date)) <= 3 * 86400) AS date_3d,
       count(*) FILTER (WHERE c.amount = t.amount AND c.merchant_key = t.merchant_key) AS amount_exact_merchant,
       count(*) FILTER (WHERE abs(c.amount - t.amount) <= 0.10 * t.amount AND c.merchant_key = t.merchant_key) AS amount_10_merchant,
       count(*) FILTER (WHERE c.amount = t.amount AND abs(date_diff('second', c.transaction_date, t.transaction_date)) <= 3 * 86400) AS amount_exact_date_3d,
       count(*) FILTER (WHERE abs(c.amount - t.amount) <= 0.10 * t.amount AND abs(date_diff('second', c.transaction_date, t.transaction_date)) <= 3 * 86400) AS amount_10_date_3d,
       count(*) FILTER (WHERE c.merchant_key = t.merchant_key AND abs(date_diff('second', c.transaction_date, t.transaction_date)) <= 3 * 86400) AS merchant_date_3d,
       count(*) FILTER (WHERE c.transaction_date > t.transaction_date) AS newer_than_target
FROM tgt2 t JOIN disp c ON c.customer_id = t.customer_id
 AND c.transaction_date BETWEEN t.session_ts - INTERVAL {WINDOW_DAYS} DAY AND t.session_ts
GROUP BY ALL""").sort_values("tid", ignore_index=True)
assert abs(len(cand) - N) < 20, len(cand)  # el redondeo por estrato da 20.001
assert (cand.amount_exact >= 1).all(), "la objetivo debe cumplir siempre su propia pista"
print(len(cand), "consultas")

20001 consultas


### 0. Sin pistas: tamaño del conjunto de candidatas

In [3]:
dist = cand.n_window.clip(upper=10).value_counts(normalize=True).sort_index().mul(100).round(1)
print("nº de candidatas en 60 días (%):"); print(dist.rename(index={10: "10+"}).to_string())
print("\nmedia:", cand.n_window.mean().round(2), "| mediana:", cand.n_window.median(),
      "| % con 1 sola:", round((cand.n_window == 1).mean() * 100, 1))
print("'la más reciente' acierta en:", round((cand.newer_than_target == 0).mean() * 100, 1), "% (sin ninguna pista)")
print("azar uniforme acierta en:", round((1 / cand.n_window).mean() * 100, 1), "%")

nº de candidatas en 60 días (%):
n_window
1      27.8
2      29.9
3      20.7
4      11.9
5       5.5
6       2.5
7       1.0
8       0.4
9       0.1
10+     0.1

media: 2.52 | mediana: 2.0 | % con 1 sola: 27.8
'la más reciente' acierta en: 71.3 % (sin ninguna pista)
azar uniforme acierta en: 54.4 %


**Interpretación:** Sin ninguna pista, la ventana de 60 días tiene 2,52 candidatas de media (mediana 2), y el 27,8 % de las consultas tiene **una sola candidata**, así que no hay nada que ordenar. Elegir "la más reciente" ya acierta el 71,3 %, frente al 54,4 % del azar. Esa ventaja de la recencia no es un dato del banco: la genera nuestro supuesto de que la sesión ocurre 0–30 días después del cargo dentro de una ventana de 60. Hay que tratarla como sesgo del generador.

### 1–4. Empates con cada combinación de pistas perfectas

In [4]:
rows = {
    "ninguna (toda la ventana)": "n_window",
    "1. monto exacto": "amount_exact",
    "2. monto ±10 %": "amount_10",
    "   solo comercio/categoría": "merchant_only",
    "   solo fecha ±3 días": "date_3d",
    "3a. monto exacto + comercio": "amount_exact_merchant",
    "3b. monto ±10 % + comercio": "amount_10_merchant",
    "4a. monto exacto + fecha ±3 d": "amount_exact_date_3d",
    "4b. monto ±10 % + fecha ±3 d": "amount_10_date_3d",
    "   comercio + fecha ±3 d": "merchant_date_3d",
}
def boot_ci(x, B=1000, seed=SEED):
    r = np.random.default_rng(seed); x = np.asarray(x, float)
    m = [x[r.integers(0, len(x), len(x))].mean() for _ in range(B)]
    return np.percentile(m, [2.5, 97.5])
tab = []
for name, col in rows.items():
    s = cand[col]; lo, hi = boot_ci(s == 1)
    tab.append({"pistas": name, "% única (regla resuelve)": round((s == 1).mean() * 100, 2),
                "IC95": f"[{lo*100:.1f}, {hi*100:.1f}]", "% ≥2 empates": round((s >= 2).mean() * 100, 2),
                "% ≥3": round((s >= 3).mean() * 100, 2), "media coincidentes": round(s.mean(), 3), "máx": s.max(),
                "top-1 si se elige al azar entre empatadas": round((1 / s).mean() * 100, 2)})
summary = pd.DataFrame(tab); print(summary.to_string(index=False))

                       pistas  % única (regla resuelve)           IC95  % ≥2 empates  % ≥3  media coincidentes  máx  top-1 si se elige al azar entre empatadas
    ninguna (toda la ventana)                     27.82   [27.2, 28.4]         72.18 42.24               2.524   12                                      54.40
              1. monto exacto                    100.00 [100.0, 100.0]          0.00  0.00               1.000    1                                     100.00
               2. monto ±10 %                     90.16   [89.7, 90.6]          9.84  0.81               1.107    4                                      94.94
      solo comercio/categoría                     82.63   [82.1, 83.2]         17.37  3.76               1.221    6                                      90.61
           solo fecha ±3 días                     85.86   [85.4, 86.3]         14.14  1.26               1.155    5                                      92.71
  3a. monto exacto + comercio                 

In [5]:
# distribución completa de empates para las pistas principales
for col in ["amount_10", "merchant_only", "date_3d", "amount_10_merchant"]:
    print(col, (cand[col].clip(upper=6).value_counts(normalize=True).sort_index() * 100).round(2).rename(index={6: "6+"}).to_dict())

amount_10 {1: 90.16, 2: 9.03, 3: 0.73, 4: 0.08}
merchant_only {1: 82.63, 2: 13.61, 3: 2.92, 4: 0.69, 5: 0.12, '6+': 0.03}
date_3d {1: 85.86, 2: 12.87, 3: 1.19, 4: 0.05, 5: 0.01}
amount_10_merchant {1: 97.94, 2: 1.99, 3: 0.06, 4: 0.0}


**Interpretación:** **El monto exacto resuelve el 100 % de las consultas** (0 empates en 20.001), porque los montos son uniformes con centavos. Aun redondeado (±10 %), deja candidata única en el 90,2 % [89,7–90,6]; con monto ±10 % + comercio sube a 97,9 %, y con monto ±10 % + fecha ±3 días, a 99,0 %. Las pistas vagas por sí solas (solo comercio: 82,6 %; solo fecha: 85,9 %) también desempatan la gran mayoría de casos. Las regiones con empates reales son estrechas: el 9,8 % con monto ±10 % y el 17,4 % con solo comercio/categoría.

## Desglose por país, canal y banda de densidad
**Pregunta:** ¿la ambigüedad se concentra en algún segmento?

In [6]:
cand["densidad"] = pd.cut(cand.n_window, [0, 1, 2, 4, 7, 100], labels=["1", "2", "3-4", "5-7", "8+"])
agg = {"n": ("tid", "size"), "cand_media": ("n_window", "mean")}
for c in ["amount_10", "merchant_only", "date_3d", "amount_10_merchant"]:
    cand[f"u_{c}"] = cand[c] == 1; agg[f"%única {c}"] = (f"u_{c}", "mean")
for by in ["country", "channel", "densidad"]:
    g = cand.groupby(by, observed=True).agg(**agg)
    g.iloc[:, 2:] = (g.iloc[:, 2:] * 100).round(1); g["cand_media"] = g.cand_media.round(2)
    print(g.to_string(), "\n")

               n  cand_media  %única amount_10  %única merchant_only  %única date_3d  %única amount_10_merchant
country                                                                                                        
Argentina   3981        2.49              90.7                  82.7            86.1                       97.9
Colombia    6007        2.51              91.1                  82.6            86.1                       98.1
México     10013        2.55              89.4                  82.6            85.6                       97.8 

             n  cand_media  %única amount_10  %única merchant_only  %única date_3d  %única amount_10_merchant
channel                                                                                                      
ATM       6007        2.51              90.5                  83.1            85.9                       97.8
App       2997        2.52              89.8                  82.2            85.1                       97.

In [7]:
# por tipo de la objetivo: compras (24 comercios) vs pagos (6 categorías) vs retiros (1 clase)
g = cand.groupby("kind").agg(**agg); g.iloc[:, 2:] = (g.iloc[:, 2:] * 100).round(1); print(g.round(2).to_string())

               n  cand_media  %única amount_10  %única merchant_only  %única date_3d  %única amount_10_merchant
kind                                                                                                           
Payment     5288        2.48              93.4                  91.7            85.9                       99.2
Purchase    7835        2.68              87.8                  95.8            84.4                       99.7
Withdrawal  6878        2.39              90.4                  60.6            87.5                       95.0


**Interpretación:** País y canal no cambian nada (diferencias ≤ 1,5 pp: son independientes, como vimos en la fase 1). Lo único que mueve la ambigüedad es la **densidad**. Con 5–7 candidatas, el monto ±10 % deja única solo el 72,8 % y el comercio solo el 66,7 %, pero esa banda es el 9 % de las consultas. Por tipo, los retiros son los más ambiguos si solo se da la categoría (60,6 %: todos los retiros son "un retiro"), y las compras los menos (95,8 %: 24 comercios).

## Sensibilidad: ¿y si la ventana incluyera todos los movimientos?
El universo disputable excluye depósitos, transferencias y ajustes. Si el banco mostrara al ranker
**todo** el extracto, ¿cambia la ambigüedad del monto ±10 %?

In [8]:
alln = q(f"""
SELECT t.transaction_id, count(*) AS n_all,
       count(*) FILTER (WHERE abs(c.amount - t.amount) <= 0.10 * t.amount) AS amount_10_all
FROM tgt2 t JOIN transactions c ON c.customer_id = t.customer_id
 AND c.transaction_date BETWEEN t.session_ts - INTERVAL {WINDOW_DAYS} DAY AND t.session_ts
GROUP BY ALL""")
print("candidatas (todos los tipos): media", alln.n_all.mean().round(2), "| mediana", alln.n_all.median())
print("% única con monto ±10 % (todos los tipos):", round((alln.amount_10_all == 1).mean() * 100, 2))

candidatas (todos los tipos): media 3.28 | mediana 3.0
% única con monto ±10 % (todos los tipos): 89.21


**Interpretación:** Mostrar al ranker todo el extracto sube las candidatas a 3,28 de media, pero apenas mueve la unicidad del monto ±10 % (89,2 % frente a 90,2 %): las transferencias y depósitos tienen rangos de monto distintos y casi nunca colisionan. Ampliar el universo no crea ambigüedad útil.

## Conclusión de la fase 2
| Pistas que da el cliente (perfectas) | % que una regla exacta resuelve sola |
|---|---|
| Monto exacto | 100,0 % |
| Monto exacto + comercio o fecha | 100,0 % |
| Monto ±10 % + fecha ±3 d | 99,0 % |
| Monto ±10 % + comercio | 97,9 % |
| Monto ±10 % | 90,2 % |
| Solo fecha ±3 d | 85,9 % |
| Solo comercio/categoría | 82,6 % |
| Ninguna (60 días) | 27,8 % (71,3 % con "la más reciente") |

Con monto + comercio casi siempre hay candidata única: **con pistas correctas el problema es trivial** y un ranker aprendido aporta poco frente a reglas. La ambigüedad natural es chica y se concentra en tres casos: sin monto, con monto redondeado en clientes con 5+ candidatas y retiros descritos solo como "un retiro". El valor está en tolerar pistas **vagas o incorrectas** (monto equivocado, fecha desplazada) y en detectar cuándo no hay candidata clara para preguntar. Eso es lo que la fase 3 simula con ruido controlado.